# Imputação e Normalização de Dados
Neste Jupyter, estarei realizando o diagnóstico estatístico para orientar minhas decisões a respeito do tratamento da base de anúncios de aluguel em São Paulo (imoveis_sao_paulo.csv). Estarei procurando por problemas de **lacunas**, variáveis que guardem valores em **diferentes métricas** e problemas de distribuição, como a existência de uma **cauda longa**.

Dessa forma, estarei usando, respectivamente, a **imputação**, para preencher dados perdidos ou inexistentes, o **escalonamento**, para normalizar as escalas numéricas e torná-las comparáveis, e também **transformações não lineares**, para alterar a geometria da distribuição e reduzir a assimetria dos dados.

In [7]:
#importando bibliotecas
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import (
    MinMaxScaler,
    PowerTransformer,
    RobustScaler,
    StandardScaler,
)
#configurando o estilo do seaborn
sns.set_theme(style="darkgrid", palette="pastel") 

In [27]:
#importando o dataset de /data

df = pd.read_csv(Path("../data/imoveis_sao_paulo/imoveis_sao_paulo.csv"))
df.head()

,bairro,zona,tier,area,rooms,parking spaces,floor,total_floors,transit_distance_min,furniture,hoa (R$),rent amount (R$),property tax (R$),total (R$)
0,Santo Amaro,Zona Sul,mid,119.0,3,0.0,11.0,15.0,3,not furnished,NaN,5050.0,590.25,6910.0
1,Vila Mariana,Zona Sul,premium,97.9,3,1.0,15.0,25.0,10,not furnished,1072.0,5060.0,833.17,6970.0
2,Ipiranga,Zona Sul,mid,45.5,1,0.0,1.0,15.0,27,furnished,520.0,2470.0,283.92,3280.0
3,Raposo Tavares,Zona Oeste,mid,180.4,4,1.0,8.0,15.0,94,not furnished,2274.0,6270.0,775.17,9320.0
4,Brooklin,Zona Sul,luxury,36.1,0,1.0,2.0,8.0,1,not furnished,570.0,2960.0,330.00,3860.0


# 1. Primeiros Passos
Antes de começar a trabalhar na base, resolvo-me por renomear as variaveis (atributos da base) para português e remover duplicatas.

In [37]:
# Novos nomes para as colunas com espaços e símbolos
novos_nomes_colunas = {
    "parking spaces": "vagas_garagem",
    "hoa (R$)": "condominio_mensal",
    "rent amount (R$)": "aluguel_mensal",
    "property tax (R$)": "iptu_mensal",
    "rooms" : "quartos",
    "floor" : "andar",
    "total_floors" :"total_andares",
    "furniture" : "mobiliado",
    "total (R$)": "total",
    "transit_distance_min" : "distancia_centro_minutos"
}
#renmeando as colunas, removendo duplicatas e resetando o index
imoveis_df = (df.rename(columns = novos_nomes_colunas)
              .drop_duplicates()
              .reset_index(drop=True)
              )

imoveis_df.head()


,bairro,zona,tier,area,quartos,vagas_garagem,andar,total_andares,distancia_centro_minutos,mobiliado,condominio_mensal,aluguel_mensal,iptu_mensal,total
0,Santo Amaro,Zona Sul,mid,119.0,3,0.0,11.0,15.0,3,not furnished,NaN,5050.0,590.25,6910.0
1,Vila Mariana,Zona Sul,premium,97.9,3,1.0,15.0,25.0,10,not furnished,1072.0,5060.0,833.17,6970.0
2,Ipiranga,Zona Sul,mid,45.5,1,0.0,1.0,15.0,27,furnished,520.0,2470.0,283.92,3280.0
3,Raposo Tavares,Zona Oeste,mid,180.4,4,1.0,8.0,15.0,94,not furnished,2274.0,6270.0,775.17,9320.0
4,Brooklin,Zona Sul,luxury,36.1,0,1.0,2.0,8.0,1,not furnished,570.0,2960.0,330.00,3860.0


### 1.1 Novo atributo
Criei um novo atributo para distinguir entre casas (1 andar) e apartamentos (mais de 1 andar).

In [38]:
col_tipo_imoveis = np.where(imoveis_df["andar"].isnull(), "casa", "apartamento")
col_mobiliado = np.where(imoveis_df["mobiliado"].str.startswith("not"),"não", "sim") # renomeie a variavel mobiliado

imoveis_df["tipo_imovel"] = col_tipo_imoveis
imoveis_df["mobiliado"] = col_mobiliado
imoveis_df.head()

,bairro,zona,tier,area,quartos,vagas_garagem,andar,total_andares,distancia_centro_minutos,mobiliado,condominio_mensal,aluguel_mensal,iptu_mensal,total,tipo_imovel
0,Santo Amaro,Zona Sul,mid,119.0,3,0.0,11.0,15.0,3,não,NaN,5050.0,590.25,6910.0,apartamento
1,Vila Mariana,Zona Sul,premium,97.9,3,1.0,15.0,25.0,10,não,1072.0,5060.0,833.17,6970.0,apartamento
2,Ipiranga,Zona Sul,mid,45.5,1,0.0,1.0,15.0,27,sim,520.0,2470.0,283.92,3280.0,apartamento
3,Raposo Tavares,Zona Oeste,mid,180.4,4,1.0,8.0,15.0,94,não,2274.0,6270.0,775.17,9320.0,apartamento
4,Brooklin,Zona Sul,luxury,36.1,0,1.0,2.0,8.0,1,não,570.0,2960.0,330.00,3860.0,apartamento
